# 1.1.2 Basic Syntax & Data Types

Covers Python's type system with real Titanic data:
- Built-in types, type coercion, string methods
- f-strings and formatting
- Dataclasses and type annotations
- Modern Python syntax (walrus, pattern matching)

In [ ]:
# Download Titanic dataset from Hugging Face Hub
import urllib.request, csv
from pathlib import Path

DATA = Path('data'); DATA.mkdir(exist_ok=True)
dest = DATA / 'titanic.csv'
if not dest.exists():
    url = 'https://huggingface.co/datasets/phihung/titanic/resolve/main/train.csv'
    try:
        urllib.request.urlretrieve(url, dest)
        print(f'Downloaded {dest.stat().st_size} bytes')
    except Exception as e:
        print(f'Fallback to synthetic: {e}')
        dest.write_text('PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked\n1,0,3,Braund Mr. Owen Harris,male,22,1,0,A/5 21171,7.25,,S\n2,1,1,Cumings Mrs. John Bradley,female,38,1,0,PC 17599,71.2833,C85,C\n')
else:
    print(f'Using cached: {dest}')

In [ ]:
# Read and explore types in the CSV
import pandas as pd

df = pd.read_csv(dest)
print('Shape:', df.shape)
print('\nDtypes:')
print(df.dtypes)
df.head()

In [ ]:
# Python type coercion examples on real data
from decimal import Decimal

sample = df.head(5)
for _, row in sample.iterrows():
    # Type conversions
    survived = bool(int(row['Survived']))  # int -> bool
    fare     = Decimal(str(round(row['Fare'], 2)))  # float -> Decimal (precise)
    age      = float(row['Age']) if pd.notna(row['Age']) else None
    name     = str(row['Name'])  # already str, but explicit
    print(f'  {name[:30]:<30}  survived={survived}  age={age}  fare={fare}')

In [ ]:
# String methods on passenger names
names = df['Name'].head(10).tolist()
print('Titles extracted from names:')
for name in names:
    # 'Braund, Mr. Owen Harris' -> 'Mr'
    title = name.split(',')[1].strip().split('.')[0].strip()
    print(f'  {name[:40]:<40} -> title={title!r}')

In [ ]:
# f-string formatting showcase
total      = len(df)
survivors  = df['Survived'].sum()
rate       = survivors / total
avg_fare   = df['Fare'].mean()

print(f'Total passengers : {total:,}')
print(f'Survivors        : {survivors:,}')
print(f'Survival rate    : {rate:.1%}')
print(f'Average fare     : £{avg_fare:>8.2f}')

# Table using f-strings
print(f'\n{"Class":>8} {"Count":>8} {"Survived":>10} {"Rate":>8}')
for cls in sorted(df['Pclass'].unique()):
    sub  = df[df['Pclass'] == cls]
    cnt  = len(sub)
    svd  = sub['Survived'].sum()
    rate_ = svd / cnt
    print(f'{cls:>8} {cnt:>8} {svd:>10} {rate_:>8.1%}')

In [ ]:
# Visualise type distribution of Age (some NaN)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Age distribution
df['Age'].dropna().hist(bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title(f'Age Distribution (N={df["Age"].notna().sum()}, NaN={df["Age"].isna().sum()})')
axes[0].set_xlabel('Age'); axes[0].set_ylabel('Count')

# Fare distribution (log scale)
df['Fare'].hist(bins=40, ax=axes[1], color='tomato', edgecolor='white')
axes[1].set_title('Fare Distribution')
axes[1].set_xlabel('Fare (£)')
plt.tight_layout(); plt.show()

In [ ]:
# Python 3.10+ structural pattern matching
def classify_passenger(row):
    match (row['Pclass'], row['Sex']):
        case (1, 'female'):
            return '1st class lady'
        case (1, 'male'):
            return '1st class gentleman'
        case (2 | 3, 'female'):
            return 'lower class woman'
        case (2 | 3, 'male'):
            return 'lower class man'
        case _:
            return 'unknown'

df['category'] = df.apply(classify_passenger, axis=1)
df.groupby('category')['Survived'].agg(['count', 'mean']).round(3)